In [31]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ["LANGCHAIN_PROJECT"] = "My First App"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [22]:
from langsmith import Client
client=Client()
dataset_name="simple EvaluationS"

dataset=client.create_dataset(dataset_name)
examples=[
         {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]

for example in examples:
    client.create_example(
        dataset_id=dataset.id,
        inputs= example["inputs"],
        outputs= example["outputs"],
    )

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

In [23]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

judge_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    
    response = judge_llm.invoke([
        ("system", eval_instructions),
        ("human", user_content)
    ])
    
    final_grade = response.content.upper()
    return "CORRECT" in final_grade

In [24]:
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [25]:
from groq import Groq
import os


raw_groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

default_instructions = "You are a helpful assistant."

def my_app(question: str, model: str = "llama-3.3-70b-versatile", instructions: str = default_instructions) -> str:
    
    response = raw_groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [26]:
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [28]:
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-ver-chatbot"
)

View the evaluation results for experiment: 'llama-ver-chatbot-fd6435a6' at:
https://smith.langchain.com/o/12808f9c-f528-41ab-a468-aefb5e447255/datasets/322c7c76-8e5d-4092-8d0d-b435208c9f73/compare?selectedSessions=eefacd9a-2ea4-47e9-89bd-74f22c045445




5it [00:21,  4.21s/it]


In [33]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs=[WebBaseLoader(url).load() for url in urls]
docs_list=[sublist for doc in docs for sublist in doc ]
chunks=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,chunk_overlap=50
).split_documents(docs_list)
embeddings=HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2"
    ,task="feature-extraction"
)

store=InMemoryVectorStore.from_documents(
    chunks,embeddings
)
retriever=store.as_retriever()
retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'HuggingFaceEndpointEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x7f0b80639d90>, search_kwargs={})

In [34]:
retriever.invoke("what is agent")

[Document(id='faf25944-4096-4e62-ad11-e3d4fdb66b3e', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [35]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile")
llm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7f0b943aa3f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f0b942b1a60>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [36]:
from langsmith import traceable

@traceable
def rag_bot(question:str):
    docs=retriever.invoke(question)
    content="".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.   
            Use the following source documents to answer the user's questions. 
            If you don't know the answer, just say that you don't know.   
            Use three sentences maximum and keep the answer concise.

            Documents:
            {content}"""
    ai_msg=llm.invoke(
        [
            {"role": "system", "content": instructions}
            ,{"role": "user", "content": question}
        ]
    )

    return {"answer": ai_msg, "documents":docs}





In [38]:
rag_bot("what is agent")['answer'].content

'An agent in the context of LLM-powered autonomous agent systems refers to a program or entity that uses a large language model (LLM) as its core controller. The agent is capable of breaking down tasks, learning from mistakes, and refining its actions through self-reflection and reflection mechanisms. It can also interact with its environment and other agents to achieve its goals.'

In [42]:
from langsmith import Client
client=Client()
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

dataset_name="Rag Tests Evaluation"
dataset=client.create_dataset(dataset_name)
for example in examples:
    client.create_example(
        dataset_id=dataset.id,
        inputs=example["inputs"],
        outputs=example["outputs"]
        
    )

